# Fine-tune LLM using LoRA and QLoRA

In [29]:
%cd /content
!rm -rf LoRA-QLoRA-studies
!git clone https://github.com/shosakaue0808/LoRA-QLoRA-studies.git
%cd /content/LoRA-QLoRA-studies

/content
Cloning into 'LoRA-QLoRA-studies'...
remote: Enumerating objects: 137, done.
remote: Counting objects: 100% (137/137), done.
remote: Compressing objects: 100% (103/103), done.
remote: Total 137 (delta 73), reused 74 (delta 29), pack-reused 0 (from 0)
Receiving objects: 100% (137/137), 208.16 KiB | 13.88 MiB/s, done.
Resolving deltas: 100% (73/73), done.
/content/LoRA-QLoRA-studies


In [30]:
!pip install transformers peft bitsandbytes python-dotenv datasets

In [31]:
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import prepare_model_for_kbit_training
from huggingface_hub import login
import torch
from torch.utils.data import DataLoader
from datasets import load_dataset
from dotenv import load_dotenv
import os
# my files
import importlib

import src.train_utils as utils
import src.data_prep as data_prep
import src.lora_layer as lora

utils = importlib.reload(utils)
data_prep=importlib.reload(data_prep)
lora = importlib.reload(lora)

## login Hugging face

In [32]:
load_dotenv()
token = os.getenv("HF_TOKEN")
login(token)

In [33]:
# Set device (use GPU if available)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cuda


# Load Model

# LoRA base

In [34]:
model_id = "meta-llama/Llama-3.2-1B"

In [35]:
tokenizer = AutoTokenizer.from_pretrained(model_id)
base_model_lora = AutoModelForCausalLM.from_pretrained(model_id)
base_model_lora.store_bit = "16bit"
tokenizer.pad_token = tokenizer.eos_token
base_model_lora.config.pad_token_id = tokenizer.pad_token_id

Loading weights:   0%|          | 0/146 [00:00<?, ?it/s]

# QLoRA base

In [ ]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16
)

In [ ]:
base_model_qlora = AutoModelForCausalLM.from_pretrained(model_id, quantization_config=bnb_config)
base_model_qlora.model_name = model_id
base_tokenizer_qlora = AutoTokenizer.from_pretrained(model_id)
base_tokenizer_qlora.pad_token = base_tokenizer_qlora.eos_token
base_model_qlora.config.pad_token_id = base_tokenizer_qlora.pad_token_id

In [ ]:
for name, p in base_model_lora.named_parameters():
    if p.requires_grad:
        print(name, p.shape)

In [36]:
#Set all weights freeze
for param in base_model_lora.parameters():
    param.requires_grad = False

# Attach LoRA layer to all linear layers in base model

In [ ]:
print(base_model_lora.model.layers)

In [37]:
lora.attach_Lora_to_Linear(base_model_lora, rank=16, alpha=16)

In [38]:
print(base_model_lora)

LlamaForCausalLM(
  (model): LlamaModel(
    (embed_tokens): Embedding(128256, 2048)
    (layers): ModuleList(
      (0-15): 16 x LlamaDecoderLayer(
        (self_attn): LlamaAttention(
          (q_proj): LayerWithLoRA(
            (base): Linear(in_features=2048, out_features=2048, bias=False)
          )
          (k_proj): LayerWithLoRA(
            (base): Linear(in_features=2048, out_features=512, bias=False)
          )
          (v_proj): LayerWithLoRA(
            (base): Linear(in_features=2048, out_features=512, bias=False)
          )
          (o_proj): LayerWithLoRA(
            (base): Linear(in_features=2048, out_features=2048, bias=False)
          )
        )
        (mlp): LlamaMLP(
          (gate_proj): LayerWithLoRA(
            (base): Linear(in_features=2048, out_features=8192, bias=False)
          )
          (up_proj): LayerWithLoRA(
            (base): Linear(in_features=2048, out_features=8192, bias=False)
          )
          (down_proj): LayerWithLoRA(
 

In [ ]:
for name, p in base_model_lora.named_parameters():
    if p.requires_grad:
        print(name, p.shape)

In [39]:
# ratio of trainable / total
total_trainable_params = sum(p.numel() for p in base_model_lora.parameters() if p.requires_grad)
total_prams= sum(p.numel() for p in base_model_lora.parameters())
print(total_trainable_params/total_prams)

0.009038820617838861


# Load dataset

In [40]:
ds = load_dataset("openai/gsm8k", "main")

In [41]:
train_ds = ds["train"]
test_ds = ds["test"]

In [42]:
train_dataset = data_prep.GSMDataset(train_ds, tokenizer)
test_dataset = data_prep.GSMDataset(test_ds, tokenizer)

Max tokens: 460
Max tokens: 416


In [43]:
from torch.utils.data import random_split

train_size = int(0.9 * len(train_dataset))
val_size = len(train_dataset) - train_size

train_dataset, val_dataset = random_split(
    train_dataset,
    [train_size, val_size]
)

train_loader = DataLoader(
    train_dataset,
    batch_size=1,
    shuffle=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=1,
    shuffle=False
)

test_loader = DataLoader(
    test_dataset,
    batch_size=1,
    shuffle=False
)

In [44]:
print(len(train_loader), len(val_loader), len(test_loader))

6725 748 1319


# Fine-tune process

In [45]:
base_model_lora = base_model_lora.to(device)
learning_rate = 1e-4
optimizer = torch.optim.AdamW(base_model_lora.parameters(), lr=learning_rate)
num_epochs = 2

In [ ]:
train(base_model_lora, optimizer, train_loader, val_loader, num_epochs, device)

epoch 1 | step 200 | train_loss 0.7129 | val_loss 0.7691
epoch 1 | step 400 | train_loss 0.6664 | val_loss 0.7330
epoch 1 | step 600 | train_loss 0.7059 | val_loss 0.7016
epoch 1 | step 800 | train_loss 0.5337 | val_loss 0.6959
epoch 1 | step 1000 | train_loss 0.9230 | val_loss 0.6740
epoch 1 | step 1200 | train_loss 0.4428 | val_loss 0.6686
epoch 1 | step 1400 | train_loss 0.5111 | val_loss 0.6603
epoch 1 | step 1600 | train_loss 0.8929 | val_loss 0.6509
epoch 1 | step 1800 | train_loss 0.3466 | val_loss 0.6517
epoch 1 | step 2000 | train_loss 0.6070 | val_loss 0.6445
epoch 1 | step 2200 | train_loss 0.5238 | val_loss 0.6376
epoch 1 | step 2400 | train_loss 0.5020 | val_loss 0.6364
epoch 1 | step 2600 | train_loss 0.4912 | val_loss 0.6339
epoch 1 | step 2800 | train_loss 0.4559 | val_loss 0.6302
epoch 1 | step 3000 | train_loss 0.4425 | val_loss 0.6247
epoch 1 | step 3200 | train_loss 0.6359 | val_loss 0.6239
epoch 1 | step 3400 | train_loss 0.6033 | val_loss 0.6193
epoch 1 | step 360